In [1]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import torch
from datasets import load_dataset
from tqdm import tqdm

from transformers import AutoImageProcessor, AutoModel, CLIPModel, CLIPProcessor

# CLIP
processor_clip = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
model_clip = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to("cuda")

# DINOv2
processor_dino = AutoImageProcessor.from_pretrained("facebook/dinov2-base", use_fast=False)
model_dino = AutoModel.from_pretrained("facebook/dinov2-base").to("cuda")

In [4]:
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
with torch.no_grad():
    ground_truth = Image.open("ground truth.png").convert("RGB")
    compare = Image.open("ground truth.png").convert("RGB")
    images = [ground_truth, compare]

    inputs = processor_clip(images=images, return_tensors="pt")
    inputs["pixel_values"] = inputs["pixel_values"].to("cuda")
    embedding = model_clip.get_image_features(**inputs)
    embedding = embedding.cpu().numpy()

    # Calculate cosine similarity
    similarity = cosine_similarity([embedding[0]], [embedding[1]])[0][0]
    print(f"Cosine similarity (CLIP): {similarity:.4f}")

    # DINOv2
    inputs = processor_dino(images=images, return_tensors="pt")
    inputs["pixel_values"] = inputs["pixel_values"].to("cuda")
    outputs = model_dino(**inputs)
    embedding = outputs.last_hidden_state
    embedding = embedding[:, 0, :].squeeze(1)
    embedding = embedding.cpu().numpy()

    # Calculate cosine similarity
    similarity = cosine_similarity([embedding[0]], [embedding[1]])[0][0]
    print(f"Cosine similarity (DINOv2): {similarity:.4f}")

Cosine similarity (CLIP): 1.0000
Cosine similarity (DINOv2): 1.0000
